# **<u>Machine Learning Model</u>**

A Random Forest multi-class classification model is selected to identify patterns in ACE National Lottery Project Grants and assess whether award funding tiers can be reliably predicted from the available features.

The objective is to assess whether the model can reliably classify awards into funding tiers and provide useful insights into patterns within ACE National Lottery Project Grants. Predictive performance and feature importance will inform the model's potential analytical value without implying causation or suitability for automated funding decisions.

#### **Target Variable**

The continuous <code>award_amount</code> variable is converted into a categorical target, <code>award_tier</code>, using the 33rd percentile (£23,750) and 67th percentile (£30,000) as boundaries. This produces three approximately balanced classes: Low (33.02%), Medium (34.41%) and High (32.58%).

#### **Features**

The model uses <code>ace_area</code>, <code>local_authority</code> and <code>main_discipline</code> as input feature to predict <code>award_tier</code> to explore geographical, disciplinary and strand patterns.

#### **Model Evaluation**

Stratified five-fold cross-validation is applied to the training data to assess model consistency while preserving the approximate distribution of funding tiers across folds. The selected model is then evaluated on the unseen test set using accuracy, macro F1-score, a confusion matrix, precision and recall.

Feature importance is analysed separately to identify which variables contribute most to the model's predictions.


</b>

## **<u>Load Libraries and Data</u>**

In [23]:
# Data manipulation
import pandas as pd
import numpy as np

# Data splitting and model evaluation
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
)
from sklearn.metrics import (
   accuracy_score,
   classification_report,
   confusion_matrix
)

# Machine learning model
from sklearn.ensemble import RandomForestClassifier

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
data = pd.read_csv('/Users/admin/Documents/Code Institute/ace_project/data/clean/model_data.csv')

# Display the first few rows of the dataset
data.head()

,ace_area,local_authority,main_discipline,award_tier
0,South West,"Bournemouth, Christchurch and Poole",Music,Medium
1,South West,Portsmouth,Theatre,Medium
2,South West,Bath and North East Somerset,Combined arts,Medium
3,South West,"Bristol, City of",Visual arts,High
4,South West,South Gloucestershire,Theatre,Low


</b>

## **<u>Model Preperation</u>**

### **Predictors and Target**

In [24]:
# Separate features and target variable
x = data.drop('award_tier', axis=1)  # Features
y = data['award_tier']  # Target variable

# Check the shape of the features and target variable
print(f"Features shape: {x.shape}")
print(f"Target variable shape: {y.shape}")

Features shape: (5452, 3)
Target variable shape: (5452,)


</b>

### **Split Train/Test Data**

The dataset is divided into training (80%) and test (20%) sets. The training set is used for model development and stratified five-fold cross-validation, while the unseen test set is reserved for final evaluation of predictive performance.

The <code>stratify=y</code> parameter preserves the approximate distribution of the three award_tier classes across both sets. Setting <code>random_state=42</code> ensures the split is reproducible. The value has no statistical significance; it was selected as a fixed seed, with a small nod to The Hitchhiker's Guide to the Galaxy.

In [25]:
# Split the dataset into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    x,                  # Predictor features
    y,                  # Target variable: award_tier
    test_size=0.2,      # Reserve 20% for final evaluation
    random_state=42,    # Ensure reproducibility
    stratify=y          # Preserve the distribution of funding tiers
)

# Validate the dimensions of the resulting datasets.
print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

Training features: (4361, 3)
Test features: (1091, 3)
Training target: (4361,)
Test target: (1091,)


</b>

## **<u>Preprocess Categorical Features</u>**

Use One-hot encoding to convert categorical predictor variables into numerical features suitable for the Random Forest classifier. A <code>ColumnTransformer</code> applies <code>OneHotEncoder</code> to the selected categorical columns, while <code>handle_unknown="ignore"</code> allows the model to process categories encountered during testing that were not present during training.

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


# List the categorical predictor variables that require one-hot encoding
categorical_cols = [
   "ace_area",
   "local_authority",
   "main_discipline"
]


# Create a preprocessing transformer for the categorical variables
preprocessor = ColumnTransformer(
   transformers=[
       (
           "categorical",
           OneHotEncoder(
               handle_unknown="ignore"
           ),
           categorical_cols
       )
   ]
)

# Fit the encoder on training data and transform the features
X_train_encoded = preprocessor.fit_transform(X_train)

# Check the transformed data dimensions
print("Original shape:", X_train.shape)
print("Encoded shape:", X_train_encoded.shape)


Original shape: (4361, 3)
Encoded shape: (4361, 309)


</b>

### **Stratified Cross-Validation Folds**

Stratified five-fold cross-validation is used during model development to assess the consistency of the Random Forest model. The training data is divided into five folds, maintaining approximately the same proportion of Low, Medium and High funding tiers in each fold. The model is trained and validated five times, with each fold used once for validation.

Shuffling randomises the data before splitting, while <code>random_state=42</code> ensures reproducibility. Cross-validation is applied exclusively to the training data, keeping the unseen test set separate for final mode evaluation.


In [27]:
# Set up cross-validation for model evaluation
from sklearn.model_selection import StratifiedKFold


# Create a stratified 5-fold cross-validation splitter
cv = StratifiedKFold(
   n_splits=5,      # Use 5 folds for cross-validation
   shuffle=True,    # Shuffle rows before splitting to reduce ordering bias
   random_state=42  # Fix random state for reproducibility
)

# Check the number of folds and the total number of training records
print("Number of folds:", cv.get_n_splits(X_train, y_train))
print("Total training records:", len(y_train))

Number of folds: 5
Total training records: 4361


</b>

## <u>**Random Forest Model Training**</u>

Train the Random Forest classifier on the dataset split for training. The preprocessing pipeline applies one-hot encoding to the categorical features before fitting the model. This allows the model to learn patterns associated with the three funding tiers.

The unseen test set remains separate during training and is reserved for the final evaluation of predictive performance.

In [28]:
from sklearn.ensemble import RandomForestClassifier

# Create the complete modelling pipeline
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42
    ))
])

# Train the final pipeline on all training records
rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['High','Low','Medium']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['ace_area','local_authority','main_discipline']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaini

</b>

## <u>**Model Evaluation**</u>

Evaluate the trained model on the unseen test dataset using accuracy, precision, recall, F1-score and a confusion matrix. These metrics assess overall predictive performance and classification accuracy across the Low, Medium and High funding tiers.

#### **Cross-Validation Result**

Assess consistency across the five folds using mean macro F1-score and standard deviation.

In [29]:
from sklearn.model_selection import cross_val_score

# Evaluate macro F1-score across five folds
cv_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro"
)

# Display cross-validation results
print("Fold macro F1-scores:", cv_scores)
print("Mean macro F1-score:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())

Fold macro F1-scores: [0.39092089 0.38419643 0.38972563 0.40128693 0.40168891]
Mean macro F1-score: 0.39356375528895443
Standard deviation: 0.006857527218052508


The Random Forest achieves a mean macro F1-score of 0.394 across five stratified validation folds, with a standard deviation of 0.007. 

The relatively low variation indicates consistent performance across folds; however, the overall score suggests limited predictive ability across the three funding tiers. Assess the model's performance further using the held-out test dataset.

</b>

#### **Train Final Model**

Train final model prior to accuracy, precision, recall and confusin matrix evaluation.

In [30]:
# Fit the pipeline on the complete training dataset
rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['High','Low','Medium']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['ace_area','local_authority','main_discipline']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaini

</b>

#### **Generate Test-Set Prediction**

Now, the RandomForest model has been trained using the entire dataset split for training instead of across the folds generate the results for the test data.

In [32]:
# Predict funding tiers for the unseen test data
y_pred = rf_pipeline.predict(X_test)

# Display the results in dataframe format for better readability
results_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

# Display the first 10 predictions alongside actual values
results_df.head(10) 

,Actual,Predicted
1896,Low,Low
1272,Low,Low
3510,High,Medium
840,Low,Low
3064,Medium,Medium
745,High,Medium
2927,Low,Low
1984,Low,Low
310,Medium,Medium
4903,Medium,High


</b>

#### **Accuracy**

Accuracy measures the proportion of test records assigned to the correct funding tier.

In [ ]:
from sklearn.metrics import accuracy_score

# Calculate the proportion of correctly classified test samples
accuracy = accuracy_score(y_test, y_pred)

# Print the model's accuracy on the unseen test set
print("Test accuracy:", accuracy)

Test accuracy: 0.39596700274977087


The Random Forest test accuracy result is 39.60%. In other words, approximately 40% of unseen records were assigned to the correct funding tier which is consistent with the results from the cross-validation evaluation result.